In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
# load processed df
from IPython.utils.capture import capture_output

with capture_output():
    %run balance_over_time.ipynb

In [3]:
features_df

,balance__mean__all,balance__median__all,balance__min__all,balance__max__all,balance__std__all,balance__pct_negative__all,balance__pct_below_100__all,balance__pct_below_500__all,n_days__all,balance__mean__30d,...,cat_39__cat_n__90d,cat_40__cat_n__90d,cat_45__cat_n__90d,cat_46__cat_n__90d,income__total__all,essentials_spend__total__all,discretionary_spend__total__all,essentials__pct_of_income__all,discretionary__pct_of_income__all,DQ_TARGET
prism_consumer_id,,,,,,,,,,,,,,,,,,,,,
0,276.961538,70.09,-1019.10,2732.86,1016.288836,0.475524,0.510490,0.580420,143.0,-497.176071,...,3.0,1.0,0.0,0.0,9340.520508,1718.160034,7332.270020,0.183947,0.784996,0.0
1,1674.533585,1758.35,-123.25,3597.09,1159.524803,0.056604,0.094340,0.301887,106.0,2671.932727,...,0.0,2.0,0.0,0.0,13414.009766,680.210022,11539.479492,0.050709,0.860256,0.0
10,-106.435115,-98.40,-1108.49,929.25,501.726601,0.595420,0.633588,0.839695,131.0,-382.525455,...,0.0,1.0,0.0,4.0,15513.070312,1527.850220,9379.980469,0.098488,0.604650,0.0
100,-3231.228909,-3752.93,-6273.18,802.40,2080.213280,0.963636,0.963636,0.981818,55.0,-4166.195000,...,0.0,0.0,0.0,0.0,24423.531250,18534.431641,200.000000,0.758876,0.008189,0.0
1000,1013.427875,615.39,-22.85,12589.57,1545.044777,0.025000,0.112500,0.437500,80.0,601.327692,...,0.0,0.0,1.0,0.0,58994.343750,17348.220703,0.000000,0.294066,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,49.549474,-44.27,-121.55,776.47,238.842211,0.631579,0.754386,0.929825,57.0,-15.570000,...,2.0,0.0,0.0,0.0,11226.839844,6032.720703,2211.799805,0.537348,0.197010,NaN
9996,172.754000,184.21,32.72,297.21,76.111644,0.000000,0.200000,1.000000,30.0,179.411000,...,0.0,0.0,0.0,0.0,0.030000,331.070038,365.749969,11035.667969,12191.665039,NaN
9997,993.787302,863.25,96.23,2334.40,558.853248,0.000000,0.015873,0.174603,63.0,939.261304,...,6.0,0.0,0.0,0.0,17007.861328,7687.240234,2637.039795,0.451982,0.155048,NaN


In [4]:
'DQ_TARGET' in features_df.columns

True

# Data Preparation

In [5]:
# keep only labeled rows
df = features_df[features_df["DQ_TARGET"].notna()].copy()

X = df.drop(columns=["DQ_TARGET"])
y = df["DQ_TARGET"].astype(int)

# Forward Selection

In [13]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SequentialFeatureSelector, SelectKBest, f_classif
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [7]:
logreg = LogisticRegression(
    penalty="l2",
    solver="liblinear",
    max_iter=2000
)

forward_selector = SequentialFeatureSelector(
    estimator=logreg,
    n_features_to_select=50,
    direction="forward",
    scoring="roc_auc",
    cv=StratifiedKFold(5),
    n_jobs=-1
)

pipeline_forward = Pipeline([
    ("scaler", StandardScaler()),
    ("sfs", forward_selector),
    ("model", logreg)
])

pipeline_forward.fit(X, y)

selected_forward = X.columns[
    pipeline_forward.named_steps["sfs"].get_support()
]

print(f'Length: {len(selected_forward)}')
print(selected_forward)

Length: 50
Index(['balance__mean__all', 'balance__median__all', 'balance__min__all',
       'balance__max__all', 'balance__std__all', 'balance__pct_below_100__all',
       'balance__pct_below_500__all', 'n_days__all', 'balance__mean__30d',
       'balance__min__30d', 'balance__std__30d', 'n_tx__30d',
       'balance__min__60d', 'balance__mean__90d', 'balance__min__90d',
       'balance__min__180d', 'balance__std__180d',
       'balance__pct_negative__180d', 'cashflow__mean_daily__180d',
       'cashflow__volatility__180d', 'debit__total__all', 'tx__max_debit__all',
       'cat_0__cat_net_total__all', 'cat_7__cat_net_total__all',
       'cat_11__cat_net_total__all', 'cat_14__cat_net_total__all',
       'cat_20__cat_net_total__all', 'cat_24__cat_net_total__all',
       'cat_28__cat_net_total__all', 'cat_37__cat_net_total__all',
       'cat_39__cat_net_total__all', 'cat_1__cat_n__all', 'cat_3__cat_n__all',
       'cat_12__cat_n__all', 'cat_17__cat_n__all', 'cat_22__cat_n__all',
       'ca

## Backward Selection

In [11]:
cv = StratifiedKFold(n_splits=2, shuffle=True, random_state=42)

logreg = LogisticRegression(
    penalty="l2",
    solver="saga",
    max_iter=1200,
    tol=5e-2,   # looser tol = faster
    n_jobs=-1
    
)

prefilter_k = 80  

backward_selector = SequentialFeatureSelector(
    estimator=logreg,
    n_features_to_select=50,   
    direction="backward",
    scoring="roc_auc",
    cv=cv,
    n_jobs=-1
)

pipeline_backward = Pipeline([
    ("prefilter", SelectKBest(score_func=f_classif, k=prefilter_k)),
    ("scaler", StandardScaler()),
    ("sfs", backward_selector),
    ("model", logreg)
])

pipeline_backward.fit(X, y)

pref_cols = X.columns[pipeline_backward.named_steps["prefilter"].get_support()]
selected_backward = pref_cols[pipeline_backward.named_steps["sfs"].get_support()]

print(f'Length: {len(selected_backward)}')
print(selected_backward)

Length: 50
Index(['balance__max__all', 'balance__std__all', 'balance__pct_negative__all',
       'balance__pct_below_100__all', 'balance__pct_below_500__all',
       'n_days__all', 'balance__mean__30d', 'n_tx__30d', 'n_days__30d',
       'balance__mean__60d', 'balance__pct_negative__60d',
       'balance__pct_negative__90d', 'balance__std__180d',
       'balance__pct_negative__180d', 'cashflow__net__180d', 'n_days__180d',
       'tx__n__all', 'credit__total__all', 'tx__max_credit__all',
       'cat_3__cat_net_total__all', 'cat_14__cat_net_total__all',
       'cat_16__cat_net_total__all', 'cat_18__cat_net_total__all',
       'cat_20__cat_net_total__all', 'cat_28__cat_net_total__all',
       'cat_31__cat_net_total__all', 'cat_35__cat_net_total__all',
       'cat_39__cat_net_total__all', 'cat_1__cat_n__all', 'cat_4__cat_n__all',
       'cat_11__cat_n__all', 'cat_12__cat_n__all', 'cat_13__cat_n__all',
       'cat_16__cat_n__all', 'cat_18__cat_n__all', 'cat_35__cat_n__all',
       'cat_46__

## Model Performance

In [14]:
df = features_df[features_df["DQ_TARGET"].notna()].copy()

y = df["DQ_TARGET"].astype(int)
X = df.drop(columns=["DQ_TARGET"]).apply(pd.to_numeric, errors="coerce").fillna(0)

In [15]:
X_all = X
X_forward = X[selected_forward]
X_backward = X[selected_backward]


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

logreg = LogisticRegression(
    penalty="l2",
    solver="liblinear",
    max_iter=2000
)

In [16]:
def eval_model(X_subset, y, name):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", logreg)
    ])
    scores = cross_validate(
        pipe,
        X_subset,
        y,
        scoring=["roc_auc", "accuracy"],
        cv=cv,
        n_jobs=-1
    )
    return {
        "Model": name,
        "Num_Features": X_subset.shape[1],
        "Avg ROC-AUC": scores["test_roc_auc"].mean(),
        "Avg Accuracy": scores["test_accuracy"].mean()
    }

In [17]:
results = [
    eval_model(X_all, y, "All Features"),
    eval_model(X_forward, y, "Forward Selected"),
    eval_model(X_backward, y, "Backward Selected"),
]

results_df = pd.DataFrame(results).sort_values("Avg ROC-AUC", ascending=False)
results_df

,Model,Num_Features,Avg ROC-AUC,Avg Accuracy
2,Backward Selected,50,0.760740,0.910051
1,Forward Selected,50,0.758230,0.909179
0,All Features,177,0.755944,0.908113
